# TSSTG Temporal Pose Dropout 미적용 학습 (seed 42)
기존 학습 ZIP을 그대로 사용합니다. 위치 1, 위치 2, 전체 결합 모델을 같은 split과 설정으로 학습하되 **temporal pose dropout만 비활성화**합니다. 좌우 반전과 XY Gaussian noise는 유지됩니다.

In [1]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
Tesla T4, 15360 MiB


## 1. 기존 업로드 파일 재사용
현재 런타임에 `/content/tsstg_full200_seed42` 폴더가 남아 있으면 바로 사용합니다. 런타임이 초기화된 경우에만 기존 `tsstg_full200_colab_seed42_ready.zip`을 다시 선택합니다.

In [3]:
from google.colab import files
!mkdir -p /content/tsstg_full200_seed42
!unzip -q -o "/content/tsstg_full200_colab_seed42_ready.zip" -d /content/tsstg_full200_seed42
%cd /content/tsstg_full200_seed42
!find . -maxdepth 2 -type f | sort | head -30

/content/tsstg_full200_seed42
./docs/TSSTG_200_EXPERIMENT_KO.md
./tsstg_pipeline/compare_models.py
./tsstg_pipeline/__init__.py
./tsstg_pipeline/train_pilot.py


In [4]:
from google.colab import files
from pathlib import Path
import os, shutil, subprocess

workdir = Path('/content/tsstg_full200_seed42')
if not (workdir / 'tsstg_pipeline' / 'train_pilot.py').exists():
    print('기존 작업 폴더가 없어 ZIP 업로드가 필요합니다.')
    uploaded = files.upload()
    archive = Path('/content') / next(iter(uploaded))
    if workdir.exists():
        shutil.rmtree(workdir)
    workdir.mkdir(parents=True)
    subprocess.run(['unzip', '-q', '-o', str(archive), '-d', str(workdir)], check=True)
else:
    print('기존 작업 폴더를 재사용합니다:', workdir)
os.chdir(workdir)
print('working directory:', Path.cwd())

기존 작업 폴더를 재사용합니다: /content/tsstg_full200_seed42
working directory: /content/tsstg_full200_seed42


In [5]:
import json
for name in ['S001_location1_compare', 'S001_location2_compare', 'S001_full200_compare']:
    summary = json.loads((Path('tsstg_binary_dataset') / name / 'dataset_summary.json').read_text())
    print(name, summary['split_counts'], 'excluded=', summary['excluded_train_event_ids'])

S001_location1_compare {'train': 80, 'validation': 10, 'test': 10} excluded= []
S001_location2_compare {'train': 79, 'validation': 10, 'test': 10} excluded= ['S001_E101']
S001_full200_compare {'train': 159, 'validation': 20, 'test': 20} excluded= ['S001_E101']


## 2. Temporal pose dropout 없이 세 모델 학습
`--temporal-dropout-probability 0`과 `--temporal-dropout-max-frames 0`으로 완전히 비활성화합니다.

In [6]:
!python -m tsstg_pipeline.train_pilot \
  --dataset tsstg_binary_dataset/S001_location1_compare \
  --output run_location1_nodropout_seed42 --head-epochs 15 --finetune-epochs 30 \
  --temporal-dropout-probability 0 --temporal-dropout-max-frames 0 --seed 42

device=cuda
Loaded 340/342 compatible tensors
/content/tsstg_full200_seed42/tsstg_pipeline/train_pilot.py:122: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss += float(loss) * len(labels)
epoch=001 train_loss=0.6910 train_acc=0.475 val_loss=0.6529 val_acc=0.600
epoch=002 train_loss=0.6482 train_acc=0.688 val_loss=0.6142 val_acc=1.000
epoch=003 train_loss=0.6272 train_acc=0.738 val_loss=0.5858 val_acc=0.900
epoch=004 train_loss=0.5834 train_acc=0.800 val_loss=0.5587 val_acc=1.000
epoch=005 train_loss=0.5519 train_acc=0.863 val_loss=0.5301 val_acc=0.900
epoch=006 train_loss=0.5295 train_acc=0.900 val_loss=0.5088 val_acc=0.900
epoch=007 train_loss=0.5159 train_acc=0.812 val_loss=0.4848 val_acc=1.000
epoch=008 train_loss=0.5386 train_acc=0.812 val_loss=0.4674 val_acc=1.000
epoch=009 train_l

In [7]:
!python -m tsstg_pipeline.train_pilot \
  --dataset tsstg_binary_dataset/S001_location2_compare \
  --output run_location2_nodropout_seed42 --head-epochs 15 --finetune-epochs 30 \
  --temporal-dropout-probability 0 --temporal-dropout-max-frames 0 --seed 42

device=cuda
Loaded 340/342 compatible tensors
/content/tsstg_full200_seed42/tsstg_pipeline/train_pilot.py:122: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss += float(loss) * len(labels)
epoch=001 train_loss=0.7235 train_acc=0.468 val_loss=0.7152 val_acc=0.500
epoch=002 train_loss=0.7028 train_acc=0.494 val_loss=0.7038 val_acc=0.500
epoch=003 train_loss=0.6572 train_acc=0.709 val_loss=0.7111 val_acc=0.400
epoch=004 train_loss=0.6328 train_acc=0.785 val_loss=0.7189 val_acc=0.500
epoch=005 train_loss=0.6057 train_acc=0.797 val_loss=0.7375 val_acc=0.500
epoch=006 train_loss=0.5955 train_acc=0.759 val_loss=0.7481 val_acc=0.500
epoch=007 train_loss=0.5808 train_acc=0.709 val_loss=0.7625 val_acc=0.600
epoch=008 train_loss=0.5405 train_acc=0.848 val_loss=0.7949 val_acc=0.500
epoch=009 train_l

In [8]:
!python -m tsstg_pipeline.train_pilot \
  --dataset tsstg_binary_dataset/S001_full200_compare \
  --output run_full200_nodropout_seed42 --head-epochs 15 --finetune-epochs 30 \
  --temporal-dropout-probability 0 --temporal-dropout-max-frames 0 --seed 42

device=cuda
Loaded 340/342 compatible tensors
/content/tsstg_full200_seed42/tsstg_pipeline/train_pilot.py:122: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss += float(loss) * len(labels)
epoch=001 train_loss=0.7063 train_acc=0.465 val_loss=0.6730 val_acc=0.500
epoch=002 train_loss=0.6552 train_acc=0.579 val_loss=0.6441 val_acc=0.700
epoch=003 train_loss=0.5994 train_acc=0.755 val_loss=0.6488 val_acc=0.750
epoch=004 train_loss=0.5617 train_acc=0.805 val_loss=0.6572 val_acc=0.800
epoch=005 train_loss=0.5528 train_acc=0.767 val_loss=0.6415 val_acc=0.750
epoch=006 train_loss=0.5399 train_acc=0.755 val_loss=0.6286 val_acc=0.750
epoch=007 train_loss=0.5160 train_acc=0.767 val_loss=0.6282 val_acc=0.750
epoch=008 train_loss=0.5051 train_acc=0.786 val_loss=0.6166 val_acc=0.750
epoch=009 train_l

## 3. 동일 Test 20 및 Test 40 평가

In [9]:
runs = {
    'location1_nodropout': 'run_location1_nodropout_seed42',
    'location2_nodropout': 'run_location2_nodropout_seed42',
    'full200_nodropout': 'run_full200_nodropout_seed42',
}
for name, run in runs.items():
    !python -m tsstg_pipeline.compare_models --dataset tsstg_binary_dataset/S001_full200_compare --binary-weights {run}/binary_tsstg_state_dict.pth --splits test --output {run}/common_test_comparison --device cuda
    !python -m tsstg_pipeline.compare_models --dataset tsstg_binary_dataset/S001_full200_compare --binary-weights {run}/binary_tsstg_state_dict.pth --splits validation test --output {run}/test40_comparison --device cuda

{
  "device": "cuda",
  "evaluated_splits": [
    "test"
  ],
  "data_files": [
    "/content/tsstg_full200_seed42/tsstg_binary_dataset/S001_full200_compare/test.npz"
  ],
  "samples": 20,
  "class_order": [
    "NON_FALL",
    "FALL"
  ],
  "confusion_matrix_convention": "rows=true, columns=predicted",
  "original_mapping": {
    "FALL": "7-class argmax == Fall Down",
    "NON_FALL": "7-class argmax is any other class"
  },
  "original_7class_tsstg": {
    "accuracy": 0.7,
    "balanced_accuracy": 0.7,
    "fall_precision": 1.0,
    "fall_recall": 0.4,
    "fall_f1": 0.5714285714285715,
    "non_fall_specificity": 1.0,
    "confusion_matrix": [
      [
        10,
        0
      ],
      [
        6,
        4
      ]
    ]
  },
  "fine_tuned_binary_tsstg": {
    "accuracy": 0.8,
    "balanced_accuracy": 0.8,
    "fall_precision": 0.8,
    "fall_recall": 0.8,
    "fall_f1": 0.8000000000000002,
    "non_fall_specificity": 0.8,
    "confusion_matrix": [
      [
        8,
        2
   

In [10]:
import pandas as pd
rows = []
for scope, subdir in [('test20', 'common_test_comparison'), ('test40', 'test40_comparison')]:
    first_report = None
    for name, run in runs.items():
        report = json.loads((Path(run) / subdir / 'comparison_metrics.json').read_text())
        first_report = first_report or report
        metric = report['fine_tuned_binary_tsstg']
        rows.append({'model': name, 'scope': scope, **{k: metric[k] for k in ['accuracy','balanced_accuracy','fall_precision','fall_recall','fall_f1','non_fall_specificity']}})
    raw = first_report['original_7class_tsstg']
    rows.append({'model': 'raw_tsstg', 'scope': scope, **{k: raw[k] for k in ['accuracy','balanced_accuracy','fall_precision','fall_recall','fall_f1','non_fall_specificity']}})
results = pd.DataFrame(rows).sort_values(['scope','model']).reset_index(drop=True)
display(results.style.format(precision=3))
results.to_csv('seed42_nodropout_model_comparison.csv', index=False, encoding='utf-8-sig')

,model,scope,accuracy,balanced_accuracy,fall_precision,fall_recall,fall_f1,non_fall_specificity
0,full200_nodropout,test20,0.800,0.800,0.800,0.800,0.800,0.800
1,location1_nodropout,test20,0.800,0.800,0.800,0.800,0.800,0.800
2,location2_nodropout,test20,0.700,0.700,0.700,0.700,0.700,0.700
3,raw_tsstg,test20,0.700,0.700,1.000,0.400,0.571,1.000
4,full200_nodropout,test40,0.850,0.850,0.850,0.850,0.850,0.850
5,location1_nodropout,test40,0.800,0.800,0.800,0.800,0.800,0.800
6,location2_nodropout,test40,0.750,0.750,0.750,0.750,0.750,0.750
7,raw_tsstg,test40,0.675,0.675,1.000,0.350,0.519,1.000


## 4. 결과 다운로드

In [11]:
!zip -qr tsstg_seed42_no_temporal_dropout_results.zip run_location1_nodropout_seed42 run_location2_nodropout_seed42 run_full200_nodropout_seed42 seed42_nodropout_model_comparison.csv
files.download('tsstg_seed42_no_temporal_dropout_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>